# NYC Congestion Pricing — SubPop Full Pipeline (Colab)

**Runtime required:** GPU (L4 recommended, T4 works for zero-shot, A100 for fine-tuning)

In Colab: Runtime → Change runtime type → L4 GPU

---
**Order of cells:**
1. Mount Drive + clone repo
2. Install dependencies
3. HuggingFace login
3b. (Optional) Download & filter SubPop pre-training data
3c. (Optional) Convert to fine-tuning format + run sequential pre-training
4. Step 2 — zero-shot baselines (GPU, ~10 min per format)
5. Step 3 — prepare fine-tuning data (CPU)
6. Step 4 — LoRA fine-tuning QA (GPU, ~30 min)
7. Step 5 — fine-tuned QA inference (GPU)
7a. Step 4b — LoRA fine-tuning BIO (GPU, ~30 min)
7b. Step 5b — fine-tuned BIO inference (GPU)
7c. Step 4c — LoRA fine-tuning PORTRAY (GPU, ~30 min)
7d. Step 5c — fine-tuned PORTRAY inference (GPU)
7e. Step 4d — LoRA fine-tuning ALL formats (GPU, ~90 min)
7f. Step 5d — fine-tuned ALL inference (GPU)
8. Step 6 — evaluation (CPU; writes descriptive_all_rows/ + test_only_fair/)
9. Save results to Drive
10. (Optional) Mistral-7B comparison run

## Cell 1 — Mount Drive & clone repo

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Subpop_replication'
DRIVE_DIR = '/content/drive/MyDrive/Subpop_replication_outputs'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Clone or pull latest code
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Twyla123/Subpop_replication.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}
print('Working directory:', os.getcwd())

Mounted at /content/drive
Cloning into '/content/Subpop_replication'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (132/132), done.
remote: Total 198 (delta 62), reused 198 (delta 62), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 1.98 MiB | 9.44 MiB/s, done.
Resolving deltas: 100% (62/62), done.
/content/Subpop_replication
Working directory: /content/Subpop_replication


## Cell 2 — Install dependencies

In [2]:
# Install GPU dependencies
!pip install -q vllm
!pip install -q peft accelerate bitsandbytes datasets fire tqdm
!pip install -q transformers>=4.40.0

# Install the SubPop library
!cd subpop-main && pip install -q -e . && cd ..

print('All dependencies installed.')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 162.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 131.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.

## Cell 3 — HuggingFace login

In [3]:
# ── Load HuggingFace token from Colab Secrets ──────────────────────────────
# In Colab: click the 🔑 key icon in the left sidebar
#           → Add new secret → Name: HF_TOKEN → Value: hf_...
# This keeps your token out of the notebook and off GitHub.
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)
print('HuggingFace login successful.')

# Primary model — Llama-3.1-8B (access approved)
# For Mistral comparison, see optional Cell 10 at the bottom.
MODEL_NAME = 'meta-llama/Llama-3.1-8B'

HuggingFace login successful.


## Cell 3b — (Optional) Download & Filter SubPop Pre-training Data

Downloads `subpop_train.jsonl` (~33 MB) from HuggingFace and keeps only the 11 waves
relevant to congestion pricing (climate, WFH/work culture, economic burden, gig economy).

**Before running:** accept the dataset terms at https://huggingface.co/datasets/jjssuh/subpop
then re-run Cell 3 (HF login) so your token is active.

In [4]:
# ── Cell 3b: Download subpop_train.jsonl and filter to relevant waves ──────
from huggingface_hub import hf_hub_download
import json, os, pandas as pd

REPO_DIR = '/content/Subpop_replication'

# 1. Download raw jsonl (~33 MB)
raw_path = hf_hub_download(
    repo_id='jjssuh/subpop',
    filename='subpop_train.jsonl',
    repo_type='dataset',
    local_dir=f'{REPO_DIR}/subpop-main/data/subpop-pretrain/raw'
)
print(f'Downloaded: {raw_path}')

# 2. Load into DataFrame
rows = []
with open(raw_path) as f:
    for line in f:
        rows.append(json.loads(line))
df = pd.DataFrame(rows)
print(f'Full dataset: {len(df):,} rows | columns: {list(df.columns)}')

# 3. Filter to 11 waves relevant to congestion pricing:
#    W67/89/102/106/108/128 = climate & environment
#    W77/94/121             = work culture / WFH / gig economy
#    W81/103                = economic burden & financial outlook
RELEVANT_WAVES = {67, 77, 81, 89, 94, 102, 103, 106, 108, 121, 128}

def extract_wave(qkey):
    try:
        return int(str(qkey).split('_W')[-1])
    except:
        return None

df['wave'] = df['qkey'].apply(extract_wave)
df_filtered = df[df['wave'].isin(RELEVANT_WAVES)].drop(columns=['wave'])

print(f'\nFiltered to {len(RELEVANT_WAVES)} waves:')
for w in sorted(RELEVANT_WAVES):
    subset = df[df['wave'] == w]
    print(f'  W{w}: {subset["qkey"].nunique()} questions, {len(subset)} rows')
print(f'\nFiltered total: {len(df_filtered):,} rows | {df_filtered["qkey"].nunique()} unique questions')

# 4. Save where prepare_finetuning_data.py expects it:
#    REPO_ROOT/data/{dataset_name}/processed/{dataset_name}.csv
out_dir = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed'
os.makedirs(out_dir, exist_ok=True)
out_path = f'{out_dir}/subpop-pretrain.csv'
df_filtered.to_csv(out_path, index=False)
print(f'\nSaved: {out_path}')

subpop_train.jsonl:   0%|          | 0.00/33.3M [00:00<?, ?B/s]

Downloaded: /content/Subpop_replication/subpop-main/data/subpop-pretrain/raw/subpop_train.jsonl
Full dataset: 71,026 rows | columns: ['qkey', 'attribute', 'group', 'question', 'options', 'ordinal', 'responses', 'refusal_rate']

Filtered to 11 waves:
  W67: 0 questions, 0 rows
  W77: 13 questions, 286 rows
  W81: 52 questions, 1144 rows
  W89: 0 questions, 0 rows
  W94: 0 questions, 0 rows
  W102: 0 questions, 0 rows
  W103: 12 questions, 264 rows
  W106: 62 questions, 1362 rows
  W108: 83 questions, 1826 rows
  W121: 31 questions, 682 rows
  W128: 89 questions, 1958 rows

Filtered total: 7,522 rows | 342 unique questions

Saved: /content/Subpop_replication/subpop-main/data/subpop-pretrain/processed/subpop-pretrain.csv


## Cell 3c — (Optional) Convert Filtered Data to Fine-tuning Format

Runs SubPop's `prepare_finetuning_data.py` on the filtered CSV to produce
`opnqa_QA_train.csv`, `opnqa_BIO_train.csv`, `opnqa_PORTRAY_train.csv` — the same
format the fine-tuning trainer expects.  Also adds a `subpop_pretrain_dataset` config
to `datasets.py` so `run_finetune.py` can find the files via `--dataset subpop_pretrain_dataset`.

In [7]:
import pathlib

REPO_DIR = '/content/Subpop_replication'

# Fix 1: __init__.py — add subpop_pretrain_dataset to registry
init_py = pathlib.Path(f'{REPO_DIR}/subpop-main/subpop/train/datasets/__init__.py')
content = init_py.read_text()
if '"subpop_pretrain_dataset"' not in content:
    content = content.replace(
        '"cms_dataset": get_custom_dataset,',
        '"cms_dataset": get_custom_dataset,\n    "subpop_pretrain_dataset": get_custom_dataset,'
    )
    content = content.replace(
        '"cms_dataset": custom_collator_no_labels,',
        '"cms_dataset": custom_collator_no_labels,\n    "subpop_pretrain_dataset": custom_collator_no_labels,'
    )
    init_py.write_text(content)
    print('Fixed __init__.py')
else:
    print('__init__.py already fixed')

# Fix 2: config_utils.py — add path-formatting elif branch
config_utils = pathlib.Path(f'{REPO_DIR}/subpop-main/subpop/train/utils/config_utils.py')
content = config_utils.read_text()
if 'subpop_pretrain_dataset' not in content:
    old = '    else:\n        raise ValueError(f"Unknown dataset: {dataset_config.dataset}")'
    new = """    elif dataset_config.dataset in ('cms_dataset', 'subpop_pretrain_dataset'):
        steering_type = train_config.steering_type
        dataset_config.train_split = dataset_config.train_split.format(steering_type = steering_type)
        dataset_config.valid_split = dataset_config.valid_split.format(steering_type = steering_type)
        dataset_config.test_split  = dataset_config.test_split.format(steering_type = steering_type)
    else:
        raise ValueError(f"Unknown dataset: {dataset_config.dataset}")"""
    content = content.replace(old, new)
    config_utils.write_text(content)
    print('Fixed config_utils.py')
else:
    print('config_utils.py already fixed')

Fixed __init__.py
Fixed config_utils.py


In [ ]:
import subprocess, sys

REPO_DIR = '/content/Subpop_replication'

result = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MODEL_NAME,
        '--dataset',                 'subpop_pretrain_dataset',
        '--steering_type',           'QA',
        '--output_dir',              'data/subpop-pretrain/checkpoints_QA/',
        '--num_epochs',              '1',          # just 1 epoch to surface the error fast
        '--early_stopping_patience', '1',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd=f'{REPO_DIR}/subpop-main',
    capture_output=True,    # ← captures stderr so we can print it
    text=True,
)

print("=== STDOUT (last 100 lines) ===")
print('\n'.join(result.stdout.splitlines()[-100:]))
print("=== STDERR (last 100 lines) ===")
print('\n'.join(result.stderr.splitlines()[-100:]))
print("=== Return code:", result.returncode)

In [5]:
# ── Cell 3c: Convert filtered data to fine-tuning format ───────────────────
import subprocess, sys, os, pathlib

REPO_DIR = '/content/Subpop_replication'

# Step 1: Run prepare_finetuning_data.py — generates opnqa_{QA,BIO,PORTRAY}_{train,val}.csv
result = subprocess.run(
    [
        sys.executable,
        'scripts/data_generation/prepare_finetuning_data.py',
        '--dataset',                      'subpop-pretrain',
        '--steer_demographics_file_path', 'data/subpopulation_metadata/demographics_61.csv',
        '--train_ratio', '0.9',
        '--val_ratio',   '0.1',
        '--test_ratio',  '0.0',
        '--seed',        '42',
    ],
    cwd=f'{REPO_DIR}/subpop-main',
    capture_output=False,
    text=True,
)
if result.returncode != 0:
    raise RuntimeError('prepare_finetuning_data.py failed — scroll up for traceback')

# Show what was produced
out_dir = f'{REPO_DIR}/subpop-main/data/subpop-pretrain/processed'
print('Generated files:')
for fname in sorted(os.listdir(out_dir)):
    if fname.endswith('.csv') and 'opnqa_' in fname:
        nrows = sum(1 for _ in open(f'{out_dir}/{fname}')) - 1
        print(f'  {fname}  ({nrows:,} rows)')

# Step 2: Add dataset config so run_finetune.py recognises --dataset subpop_pretrain_dataset
datasets_py = pathlib.Path(f'{REPO_DIR}/subpop-main/subpop/train/configs/datasets.py')
config_block = '''

# ── SubPop Pre-training (filtered relevant waves for congestion pricing) ─────
@dataclass
class subpop_pretrain_dataset:
    dataset: str = "subpop_pretrain_dataset"
    file: str = "subpop/train/datasets/opinionqa_dataset.py:get_preprocessed_opinionqa_ce_or_wd_loss"
    train_split: str = "data/subpop-pretrain/processed/opnqa_{steering_type}_train.csv"
    valid_split: str = "data/subpop-pretrain/processed/opnqa_{steering_type}_val.csv"
    test_split:  str = "data/subpop-pretrain/processed/opnqa_{steering_type}_test.csv"
'''

content = datasets_py.read_text()
if 'subpop_pretrain_dataset' not in content:
    datasets_py.write_text(content + config_block)
    print('\nAdded subpop_pretrain_dataset config to datasets.py')
else:
    print('\nsubpop_pretrain_dataset config already present — no change needed')

# Step 3: Run sequential pre-training on the filtered subpop data
# This fine-tunes the base LLM on survey response dynamics across relevant domains
# BEFORE the CMS-specific fine-tuning in Cells 6-7e.
result_pt = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MODEL_NAME,
        '--dataset',                 'subpop_pretrain_dataset',
        '--steering_type',           'QA',
        '--output_dir',              'data/subpop-pretrain/checkpoints_QA/',
        '--num_epochs',              '10',          # fewer epochs — just domain warmup
        '--early_stopping_patience', '3',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd=f'{REPO_DIR}/subpop-main',
    capture_output=False,
    text=True,
)
if result_pt.returncode != 0:
    raise RuntimeError('SubPop pre-training FAILED — scroll up for traceback')
print('SubPop pre-training complete.')
print('Next: run Cells 6-7e for CMS fine-tuning (those cells start from the base model).')
print('To use the pre-trained checkpoint as CMS starting point, note the checkpoint path above.')

Generated files:
  opnqa_ALL_test.csv  (0 rows)
  opnqa_ALL_train.csv  (282,907 rows)
  opnqa_ALL_val.csv  (30,552 rows)
  opnqa_BIO_test.csv  (0 rows)
  opnqa_BIO_train.csv  (77,987 rows)
  opnqa_BIO_val.csv  (8,382 rows)
  opnqa_PORTRAY_test.csv  (0 rows)
  opnqa_PORTRAY_train.csv  (71,213 rows)
  opnqa_PORTRAY_val.csv  (7,634 rows)
  opnqa_QA_test.csv  (0 rows)
  opnqa_QA_train.csv  (133,707 rows)
  opnqa_QA_val.csv  (14,536 rows)

Added subpop_pretrain_dataset config to datasets.py


RuntimeError: SubPop pre-training FAILED — scroll up for traceback

## Cell 4 — Step 2: Zero-shot baselines (GPU, ~10 min per format)
Produces `distributions_QA.csv`, `distributions_BIO.csv`, `distributions_PORTRAY.csv`

In [ ]:
!python step2_vllm_baselines.py \
    --mode zeroshot \
    --model_name {MODEL_NAME} \
    --demographics_csv  approach2_outputs/cms/cms_demographics.csv \
    --steering_json     approach2_outputs/cms/cms_steering_prompts.json \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --output_dir        approach2_outputs/cms \
    --formats QA BIO PORTRAY

In [ ]:
# Statistical bounds — uses WD for ordinal questions, TVD for nominal questions
# --questions_json provides the is_ordinal flag per question
!python step2_vllm_baselines.py \
    --mode bounds \
    --ground_truth_csv  approach2_outputs/cms/cms_survey_distributions.csv \
    --questions_json    approach2_outputs/cms/cms_questions.json \
    --output_dir        approach2_outputs/cms

## Cell 5 — Step 3: Prepare fine-tuning data (CPU, ~1 min)
Produces `cms_QA_train.csv`, `cms_BIO_train.csv`, etc. in `finetuning_data/`

In [ ]:
for fmt in ['QA', 'BIO', 'PORTRAY', 'ALL']:
    pass  # generated together in one call below

!python step3_prepare_finetuning_data.py \
    --distributions_csv   approach2_outputs/cms/cms_survey_distributions.csv \
    --demographics_csv    approach2_outputs/cms/cms_demographics.csv \
    --steering_json       approach2_outputs/cms/cms_steering_prompts.json \
    --question_split_json approach2_outputs/cms/cms_question_split.json \
    --output_dir          approach2_outputs/cms/finetuning_data \
    --formats QA BIO PORTRAY ALL

## Cell 6 — Step 4: LoRA Fine-tuning (GPU, ~30 min on L4)

Run once per steering format. Start with QA (matches SubPop paper).

Checkpoints saved to `approach2_outputs/cms/checkpoints_QA/`

In [ ]:
import subprocess, sys

REPO_DIR = '/content/Subpop_replication'   # absolute — works after any runtime restart

result = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MODEL_NAME,
        '--dataset',                 'cms_dataset',
        '--steering_type',           'QA',
        '--output_dir',              '../approach2_outputs/cms/checkpoints_QA/',
        '--num_epochs',              '50',
        '--early_stopping_patience', '3',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd=f'{REPO_DIR}/subpop-main',   # absolute cwd — no os.chdir needed
    capture_output=False,
    text=True,
)

if result.returncode == 0:
    print('QA fine-tuning complete.')
else:
    raise RuntimeError(
        f'QA fine-tuning FAILED (exit code {result.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

In [ ]:
import glob, os, pathlib, subprocess, sys

# ── Locate QA checkpoint ─────────────────────────────────────────────────────
# run_finetune.py appends a timestamp, so checkpoint lives at
#   approach2_outputs/cms/checkpoints_QA/<timestamp>/
checkpoints = sorted(glob.glob('approach2_outputs/cms/checkpoints_QA/*'))
if not checkpoints:
    raise FileNotFoundError(
        'No checkpoint found under approach2_outputs/cms/checkpoints_QA/\n'
        'Did Cell 6 (QA fine-tuning) complete successfully?'
    )
checkpoint_path = checkpoints[-1]
print(f'Using checkpoint: {checkpoint_path}')

# ── Run inference (hardened: subprocess.run + explicit failure check) ────────
TEST_CSV   = 'approach2_outputs/cms/finetuning_data/cms_QA_test.csv'
LORA_NAME  = 'cms_QA'
INFER_DIR  = 'approach2_outputs/cms/'   # trailing slash required by run_inference.py

result_infer = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_inference.py',
        '--input_paths',             f'../{TEST_CSV}',
        '--output_dir',              f'../{INFER_DIR}',
        '--base_model_name_or_path', MODEL_NAME,
        '--lora_path',               f'../{checkpoint_path}',
        '--lora_name',               LORA_NAME,
    ],
    cwd='subpop-main',    # run from subpop-main/ without mutating the notebook cwd
    capture_output=False,
    text=True,
)
if result_infer.returncode != 0:
    raise RuntimeError(
        f'QA inference FAILED (exit code {result_infer.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

# ── Rename output to canonical name expected by step 6 ──────────────────────
# run_inference.py writes:  {output_dir}{input_stem}_{lora_name}.csv
raw_stem  = pathlib.Path(TEST_CSV).stem                       # cms_QA_test
raw_out   = f'{INFER_DIR}{raw_stem}_{LORA_NAME}.csv'          # .../cms_QA_test_cms_QA.csv
canonical = 'approach2_outputs/cms/results_finetuned_QA.csv'

if os.path.exists(raw_out):
    os.rename(raw_out, canonical)
    print(f'Renamed: {raw_out} → {canonical}')
else:
    # Show what was actually written so the user can diagnose
    print(f'WARNING: expected output {raw_out!r} not found.')
    print('Files matching cms_QA_test* in output dir:')
    for f in sorted(os.listdir(INFER_DIR)):
        if 'cms_QA_test' in f:
            print(f'  {f}')
    raise FileNotFoundError(
        f'QA inference did not produce the expected output file.\n'
        f'Expected: {raw_out}'
    )

print('QA inference complete.')

## Cell 7a — Step 4b: Fine-tune with BIO steering format (GPU, ~30 min on L4)

BIO is the strongest zero-shot format (WD ≈ 0.245 vs QA 0.253) so fine-tuning it provides
the best chance of improvement.  Run after Cell 6 (QA fine-tuning) completes.

Checkpoints saved to `approach2_outputs/cms/checkpoints_BIO/`

In [ ]:
import subprocess, sys

REPO_DIR = '/content/Subpop_replication'   # absolute — works after any runtime restart

result_bio = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MODEL_NAME,
        '--dataset',                 'cms_dataset',
        '--steering_type',           'BIO',
        '--output_dir',              '../approach2_outputs/cms/checkpoints_BIO/',
        '--num_epochs',              '50',
        '--early_stopping_patience', '3',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd=f'{REPO_DIR}/subpop-main',   # absolute cwd — no os.chdir needed
    capture_output=False,
    text=True,
)

if result_bio.returncode == 0:
    print('BIO fine-tuning complete.')
else:
    raise RuntimeError(
        f'BIO fine-tuning FAILED (exit code {result_bio.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

In [ ]:
import glob, os, pathlib, subprocess, sys

# ── Locate BIO checkpoint ────────────────────────────────────────────────────
checkpoints_bio = sorted(glob.glob('approach2_outputs/cms/checkpoints_BIO/*'))
if not checkpoints_bio:
    raise FileNotFoundError(
        'No checkpoint found under approach2_outputs/cms/checkpoints_BIO/\n'
        'Did Cell 7a (BIO fine-tuning) complete successfully?'
    )
checkpoint_path_bio = checkpoints_bio[-1]
print(f'Using checkpoint: {checkpoint_path_bio}')

# ── Run inference (hardened: subprocess.run + explicit failure check) ────────
TEST_CSV_BIO  = 'approach2_outputs/cms/finetuning_data/cms_BIO_test.csv'
LORA_NAME_BIO = 'cms_BIO'
INFER_DIR_BIO = 'approach2_outputs/cms/'   # trailing slash required by run_inference.py

result_infer_bio = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_inference.py',
        '--input_paths',             f'../{TEST_CSV_BIO}',
        '--output_dir',              f'../{INFER_DIR_BIO}',
        '--base_model_name_or_path', MODEL_NAME,
        '--lora_path',               f'../{checkpoint_path_bio}',
        '--lora_name',               LORA_NAME_BIO,
    ],
    cwd='subpop-main',    # run from subpop-main/ without mutating the notebook cwd
    capture_output=False,
    text=True,
)
if result_infer_bio.returncode != 0:
    raise RuntimeError(
        f'BIO inference FAILED (exit code {result_infer_bio.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

# ── Rename output to canonical name expected by step 6 ──────────────────────
# run_inference.py writes:  {output_dir}{input_stem}_{lora_name}.csv
raw_stem_bio  = pathlib.Path(TEST_CSV_BIO).stem                            # cms_BIO_test
raw_out_bio   = f'{INFER_DIR_BIO}{raw_stem_bio}_{LORA_NAME_BIO}.csv'      # .../cms_BIO_test_cms_BIO.csv
canonical_bio = 'approach2_outputs/cms/results_finetuned_BIO.csv'

if os.path.exists(raw_out_bio):
    os.rename(raw_out_bio, canonical_bio)
    print(f'Renamed: {raw_out_bio} → {canonical_bio}')
else:
    print(f'WARNING: expected output {raw_out_bio!r} not found.')
    print('Files matching cms_BIO_test* in output dir:')
    for f in sorted(os.listdir(INFER_DIR_BIO)):
        if 'cms_BIO_test' in f:
            print(f'  {f}')
    raise FileNotFoundError(
        f'BIO inference did not produce the expected output file.\n'
        f'Expected: {raw_out_bio}'
    )

print('BIO inference complete.')

## Cell 7c — Step 4c: Fine-tune with PORTRAY steering format (GPU, ~30 min on L4)

PORTRAY uses "Answer as if you were a [group] person" framing (vs BIO's first-person description).
Run after BIO fine-tuning (Cell 7a) completes, or independently.

Checkpoints saved to `approach2_outputs/cms/checkpoints_PORTRAY/`

In [ ]:
import subprocess, sys

REPO_DIR = '/content/Subpop_replication'   # absolute — works after any runtime restart

result_portray = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MODEL_NAME,
        '--dataset',                 'cms_dataset',
        '--steering_type',           'PORTRAY',
        '--output_dir',              '../approach2_outputs/cms/checkpoints_PORTRAY/',
        '--num_epochs',              '50',
        '--early_stopping_patience', '3',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd=f'{REPO_DIR}/subpop-main',
    capture_output=False,
    text=True,
)

if result_portray.returncode == 0:
    print('PORTRAY fine-tuning complete.')
else:
    raise RuntimeError(
        f'PORTRAY fine-tuning FAILED (exit code {result_portray.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

In [ ]:
import glob, os, pathlib, subprocess, sys

# ── Locate PORTRAY checkpoint ────────────────────────────────────────────────
checkpoints_portray = sorted(glob.glob('approach2_outputs/cms/checkpoints_PORTRAY/*'))
if not checkpoints_portray:
    raise FileNotFoundError(
        'No checkpoint found under approach2_outputs/cms/checkpoints_PORTRAY/\n'
        'Did Cell 7c (PORTRAY fine-tuning) complete successfully?'
    )
checkpoint_path_portray = checkpoints_portray[-1]
print(f'Using checkpoint: {checkpoint_path_portray}')

TEST_CSV_PORTRAY  = 'approach2_outputs/cms/finetuning_data/cms_PORTRAY_test.csv'
LORA_NAME_PORTRAY = 'cms_PORTRAY'
INFER_DIR_PORTRAY = 'approach2_outputs/cms/'

result_infer_portray = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_inference.py',
        '--input_paths',             f'../{TEST_CSV_PORTRAY}',
        '--output_dir',              f'../{INFER_DIR_PORTRAY}',
        '--base_model_name_or_path', MODEL_NAME,
        '--lora_path',               f'../{checkpoint_path_portray}',
        '--lora_name',               LORA_NAME_PORTRAY,
    ],
    cwd='subpop-main',
    capture_output=False,
    text=True,
)
if result_infer_portray.returncode != 0:
    raise RuntimeError(
        f'PORTRAY inference FAILED (exit code {result_infer_portray.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

raw_stem_portray  = pathlib.Path(TEST_CSV_PORTRAY).stem                                   # cms_PORTRAY_test
raw_out_portray   = f'{INFER_DIR_PORTRAY}{raw_stem_portray}_{LORA_NAME_PORTRAY}.csv'      # .../cms_PORTRAY_test_cms_PORTRAY.csv
canonical_portray = 'approach2_outputs/cms/results_finetuned_PORTRAY.csv'

if os.path.exists(raw_out_portray):
    os.rename(raw_out_portray, canonical_portray)
    print(f'Renamed: {raw_out_portray} → {canonical_portray}')
else:
    print(f'WARNING: expected output {raw_out_portray!r} not found.')
    print('Files matching cms_PORTRAY_test* in output dir:')
    for f in sorted(os.listdir(INFER_DIR_PORTRAY)):
        if 'cms_PORTRAY_test' in f:
            print(f'  {f}')
    raise FileNotFoundError(
        f'PORTRAY inference did not produce the expected output file.\n'
        f'Expected: {raw_out_portray}'
    )

print('PORTRAY inference complete.')

## Cell 7e — Step 4d: Fine-tune with ALL formats combined (GPU, ~90 min on L4)

ALL trains on `cms_ALL_train.csv` which concatenates QA + BIO + PORTRAY training rows (~3× more rows than any single format).
Training on all 3 prompt formats at once acts as regularization and may reduce overfitting.
Evaluated on QA test set for direct comparison with the QA-only fine-tuned model.

Checkpoints saved to `approach2_outputs/cms/checkpoints_ALL/`

In [ ]:
import subprocess, sys

REPO_DIR = '/content/Subpop_replication'   # absolute — works after any runtime restart

# ALL trains on cms_ALL_{train,val,test}.csv which combines QA + BIO + PORTRAY prompts.
# With 3x more rows the val loss is more stable, so we use patience=5 here.
result_all = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MODEL_NAME,
        '--dataset',                 'cms_dataset',
        '--steering_type',           'ALL',
        '--output_dir',              '../approach2_outputs/cms/checkpoints_ALL/',
        '--num_epochs',              '50',
        '--early_stopping_patience', '5',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd=f'{REPO_DIR}/subpop-main',
    capture_output=False,
    text=True,
)

if result_all.returncode == 0:
    print('ALL fine-tuning complete.')
else:
    raise RuntimeError(
        f'ALL fine-tuning FAILED (exit code {result_all.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

In [ ]:
import glob, os, pathlib, subprocess, sys

# ── Locate ALL checkpoint ────────────────────────────────────────────────────
checkpoints_all = sorted(glob.glob('approach2_outputs/cms/checkpoints_ALL/*'))
if not checkpoints_all:
    raise FileNotFoundError(
        'No checkpoint found under approach2_outputs/cms/checkpoints_ALL/\n'
        'Did Cell 7e (ALL fine-tuning) complete successfully?'
    )
checkpoint_path_all = checkpoints_all[-1]
print(f'Using checkpoint: {checkpoint_path_all}')

# Evaluate on QA test set — canonical comparison with other fine-tuned models.
# The ALL-trained model should handle any prompt format; QA is the standard baseline.
TEST_CSV_ALL  = 'approach2_outputs/cms/finetuning_data/cms_QA_test.csv'
LORA_NAME_ALL = 'cms_ALL'
INFER_DIR_ALL = 'approach2_outputs/cms/'

result_infer_all = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_inference.py',
        '--input_paths',             f'../{TEST_CSV_ALL}',
        '--output_dir',              f'../{INFER_DIR_ALL}',
        '--base_model_name_or_path', MODEL_NAME,
        '--lora_path',               f'../{checkpoint_path_all}',
        '--lora_name',               LORA_NAME_ALL,
    ],
    cwd='subpop-main',
    capture_output=False,
    text=True,
)
if result_infer_all.returncode != 0:
    raise RuntimeError(
        f'ALL inference FAILED (exit code {result_infer_all.returncode}).\n'
        'Scroll up to see the full traceback above.'
    )

raw_stem_all  = pathlib.Path(TEST_CSV_ALL).stem                          # cms_QA_test
raw_out_all   = f'{INFER_DIR_ALL}{raw_stem_all}_{LORA_NAME_ALL}.csv'     # .../cms_QA_test_cms_ALL.csv
canonical_all = 'approach2_outputs/cms/results_finetuned_ALL.csv'

if os.path.exists(raw_out_all):
    os.rename(raw_out_all, canonical_all)
    print(f'Renamed: {raw_out_all} → {canonical_all}')
else:
    print(f'WARNING: expected output {raw_out_all!r} not found.')
    print('Files matching cms_QA_test* in output dir:')
    for f in sorted(os.listdir(INFER_DIR_ALL)):
        if 'cms_QA_test' in f:
            print(f'  {f}')
    raise FileNotFoundError(
        f'ALL inference did not produce the expected output file.\n'
        f'Expected: {raw_out_all}'
    )

print('ALL inference complete.')

## Cell 8 — Step 6: Evaluation (CPU, ~1 min)

In [ ]:
!python step6_full_evaluation.py \
    --ground_truth_csv    approach2_outputs/cms/cms_survey_distributions.csv \
    --questions_json      approach2_outputs/cms/cms_questions.json \
    --question_split_json approach2_outputs/cms/cms_question_split.json \
    --predictions_dir     approach2_outputs/cms \
    --weights_csv         approach2_outputs/cms/cms_subgroup_weights.csv \
    --output_dir          approach2_outputs/cms/evaluation

import os
for f in sorted(os.listdir('approach2_outputs/cms/evaluation')):
    print(' ', f)


## Cell 9 — Save all results to Drive

In [ ]:
import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/Subpop_replication_outputs'

# Copy everything in approach2_outputs/cms to Drive
src = 'approach2_outputs/cms'
dst = os.path.join(DRIVE_DIR, 'cms')
if os.path.exists(dst):
    shutil.rmtree(dst)
shutil.copytree(src, dst)

print(f'Results saved to Google Drive: {dst}')
print('Files saved:')
for root, dirs, files in os.walk(dst):
    dirs[:] = [d for d in dirs if d != 'finetuning_data']  # skip large CSVs
    for f in files:
        rel = os.path.relpath(os.path.join(root, f), dst)
        print(' ', rel)

In [ ]:
!cd /content/Subpop_replication && git pull

---
## (Optional) Cell 10 — Run Mistral-7B for model comparison

Skip this if you only need Llama results.

Re-runs zero-shot + fine-tuning + inference with `mistralai/Mistral-7B-v0.1` and saves
results under `approach2_outputs/cms_mistral/` so the two model runs stay separate.

**Why bother?**  Comparing Llama vs Mistral shows whether the SubPop methodology is
model-agnostic — a meaningful finding for the paper.

In [ ]:
import os, glob, pathlib, subprocess, sys

MISTRAL     = 'mistralai/Mistral-7B-v0.1'
MISTRAL_OUT = 'approach2_outputs/cms_mistral'
os.makedirs(MISTRAL_OUT, exist_ok=True)

# ── Zero-shot baselines ──────────────────────────────────────────────────────
result_zs = subprocess.run(
    [
        sys.executable, 'step2_vllm_baselines.py',
        '--mode', 'zeroshot',
        '--model_name',        MISTRAL,
        '--demographics_csv',  'approach2_outputs/cms/cms_demographics.csv',
        '--steering_json',     'approach2_outputs/cms/cms_steering_prompts.json',
        '--questions_json',    'approach2_outputs/cms/cms_questions.json',
        '--output_dir',        MISTRAL_OUT,
        '--formats', 'QA', 'BIO', 'PORTRAY',
    ],
    capture_output=False, text=True,
)
if result_zs.returncode != 0:
    raise RuntimeError(f'Mistral zero-shot FAILED (exit code {result_zs.returncode}). Scroll up for traceback.')

# ── LoRA fine-tuning ─────────────────────────────────────────────────────────
result_ft = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_finetune.py',
        '--model_name',              MISTRAL,
        '--dataset',                 'cms_dataset',
        '--steering_type',           'QA',
        '--output_dir',              f'../{MISTRAL_OUT}/checkpoints_QA/',
        '--num_epochs',              '50',
        '--early_stopping_patience', '5',
        '--batch_size_training',     '4',
        '--lr',                      '2e-4',
        '--use_peft',
        '--quantization',
        '--loss_function_type',      'ce',
        '--enable_fsdp=False',
        '--use_wandb=False',
    ],
    cwd='subpop-main', capture_output=False, text=True,
)
if result_ft.returncode != 0:
    raise RuntimeError(f'Mistral fine-tuning FAILED (exit code {result_ft.returncode}). Scroll up for traceback.')

# ── Inference (hardened: subprocess.run + explicit failure check) ────────────
checkpoints = sorted(glob.glob(f'{MISTRAL_OUT}/checkpoints_QA/*'))
if not checkpoints:
    raise FileNotFoundError(f'No Mistral checkpoint found under {MISTRAL_OUT}/checkpoints_QA/')
checkpoint_path = checkpoints[-1]
print(f'Using checkpoint: {checkpoint_path}')

TEST_CSV  = 'approach2_outputs/cms/finetuning_data/cms_QA_test.csv'
LORA_NAME = 'cms_mistral_QA'
INFER_DIR = f'{MISTRAL_OUT}/'

result_infer = subprocess.run(
    [
        sys.executable, 'scripts/experiment/run_inference.py',
        '--input_paths',             f'../{TEST_CSV}',
        '--output_dir',              f'../{INFER_DIR}',
        '--base_model_name_or_path', MISTRAL,
        '--lora_path',               f'../{checkpoint_path}',
        '--lora_name',               LORA_NAME,
    ],
    cwd='subpop-main', capture_output=False, text=True,
)
if result_infer.returncode != 0:
    raise RuntimeError(f'Mistral inference FAILED (exit code {result_infer.returncode}). Scroll up for traceback.')

# ── Rename to canonical name ─────────────────────────────────────────────────
raw_stem  = pathlib.Path(TEST_CSV).stem
raw_out   = f'{INFER_DIR}{raw_stem}_{LORA_NAME}.csv'
canonical = f'{MISTRAL_OUT}/results_finetuned_QA.csv'
if os.path.exists(raw_out):
    os.rename(raw_out, canonical)
    print(f'Renamed: {raw_out} → {canonical}')
else:
    raise FileNotFoundError(
        f'Mistral inference did not produce expected output.\nExpected: {raw_out}'
    )

# ── Evaluation ───────────────────────────────────────────────────────────────
result_eval = subprocess.run(
    [
        sys.executable, 'step6_full_evaluation.py',
        '--ground_truth_csv',    'approach2_outputs/cms/cms_survey_distributions.csv',
        '--questions_json',      'approach2_outputs/cms/cms_questions.json',
        '--question_split_json', 'approach2_outputs/cms/cms_question_split.json',
        '--predictions_dir',     MISTRAL_OUT,
        '--weights_csv',         'approach2_outputs/cms/cms_subgroup_weights.csv',
        '--output_dir',          f'{MISTRAL_OUT}/evaluation',
    ],
    capture_output=False, text=True,
)
if result_eval.returncode != 0:
    raise RuntimeError(f'Mistral evaluation FAILED (exit code {result_eval.returncode}). Scroll up for traceback.')

print('Mistral run complete. Results in', MISTRAL_OUT)